In [14]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
import joblib



In [4]:
daten = load_wine()

In [10]:
X,y = daten.data, daten.target

In [11]:
X

array([[1.423e+01, 1.710e+00, 2.430e+00, ..., 1.040e+00, 3.920e+00,
        1.065e+03],
       [1.320e+01, 1.780e+00, 2.140e+00, ..., 1.050e+00, 3.400e+00,
        1.050e+03],
       [1.316e+01, 2.360e+00, 2.670e+00, ..., 1.030e+00, 3.170e+00,
        1.185e+03],
       ...,
       [1.327e+01, 4.280e+00, 2.260e+00, ..., 5.900e-01, 1.560e+00,
        8.350e+02],
       [1.317e+01, 2.590e+00, 2.370e+00, ..., 6.000e-01, 1.620e+00,
        8.400e+02],
       [1.413e+01, 4.100e+00, 2.740e+00, ..., 6.100e-01, 1.600e+00,
        5.600e+02]], shape=(178, 13))

In [6]:
X_rest, X_te, y_rest, y_te = train_test_split(X, y, random_state=42, test_size=0.2)

In [7]:
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = GridSearchCV(pipe, param_grid={
    "kneighborsclassifier__n_neighbors": [1,3,5,7,9,15,25,51],
    "kneighborsclassifier__weights": ["uniform", "distance"]
}, cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring="accuracy", return_train_score=True)

grid.fit(X_rest, y_rest)
print("best params", grid.best_params_)
print("best score", grid.best_score_)

best params {'kneighborsclassifier__n_neighbors': 25, 'kneighborsclassifier__weights': 'uniform'}
best score 0.9716748768472907


In [8]:
res = pd.DataFrame(grid.cv_results_)

tab = res[["param_kneighborsclassifier__n_neighbors", "param_kneighborsclassifier__weights", "mean_train_score", "mean_test_score", "std_test_score","rank_test_score"]]
tab.columns = ["k", "weights", "train", "validation", "sd","rang"]
tab.sort_values("rang").round(3)

,k,weights,train,validation,sd,rang
12,25,uniform,0.968,0.972,0.027,1
13,25,distance,1.000,0.972,0.027,1
15,51,distance,1.000,0.965,0.022,3
3,3,distance,1.000,0.965,0.032,4
2,3,uniform,0.968,0.965,0.032,4
11,15,distance,1.000,0.958,0.015,6
5,5,distance,1.000,0.958,0.027,6
14,51,uniform,0.958,0.951,0.027,8
1,1,distance,1.000,0.951,0.028,9
9,9,distance,1.000,0.951,0.028,9


In [12]:
print("Test accuracy", grid.score(X_te, y_te))

Test accuracy 0.9722222222222222


In [16]:
joblib.dump(grid.best_estimator_, "models/wine.joblib")

['models/wine.joblib']